In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 



In [2]:
class CausalConv1d(nn.Module): 
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int = 1): 
        super().__init__() 
        self.pad = (kernel_size - 1) * dilation 
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation) 
    
    def forward(self, x): 
        # x: (batch, in_ch, time) 
        x = F.pad(x, (self.pad, 0)) 
        return self.conv(x) 

x = torch.randn(2, 4, 100) 
conv = CausalConv1d(4, 8, kernel_size=3, dilation=4) 
y = conv(x) 
print(y.shape)

torch.Size([2, 8, 100])


In [ ]:
# weighted norm causal conv 1d 
class WeightedNormCausalConv1d(nn.Module): 
    """ 
    causal 1d conv with weight normalization built in 
    instead of storing a single weight tensor w, we store: 
        v: same shape as conv kernel (out_ch, in_ch, k) 
        g: a scalar per output channel (out_ch, 1, 1) 
    and comput w = g * v / ||v|| on every forward pass 

    This decouples the magnitude (g) from the direction (v / ||v||) of the weights
    which empirically helps deep nets train faster and more stable 
    """
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int = 1): 
        super().__init__() 
        self.pad = (kernel_size - 1) * dilation 
        self.dilation = dilation 
        self.kernel_size = kernel_size 

        # v: direction parameter (same shape as Conv1d weight tensor) 
        # small random init so we don't start at 0 - would kill gradient flow 
        self.v = nn.Parameter(torch.randn(out_ch, in_ch, kernel_size) * 0.1) 

        # g: magnitude parameter. one scalar output per channel 
        # init to ||v|| at construction time so effective weight starts at v 
        with torch.no_grad(): 
            v_norm_init = self.v.norm(dim=(1, 2), keepdim=True) 
        self.g = nn.Parameter(v_norm_init.clone()) 
        # bias - same as normal conv 1 output per channel 
        self.bias = nn.Parameter(torch.zeros(out_ch))
    
    def forward(self, x): 
        # effective weight: w = g * v / ||v|| 
        # norm is per output channel taken over (in_ch, kernel) dims together 
        v_norm = self.v.norm(dim=(1, 2), keepdim=True) 
        w = self.g * self. v (v_norm + 1e-8) 

        # causal padding then convolve wit hw 
        x = F.pad(x, (self.pad, 0)) 
        return F.conv1d(x, w, self.bias, dilation=self.dilation) 
    